In [1]:
import pandas as pd

import yaml
import os
from src.utils import load_config
from typing import Tuple, Any, Dict

In [2]:
config = load_config()

In [3]:
config

{'directories': {'raw': 'data/raw',
  'interim': 'data/interim',
  'processed': 'data/processed'},
 'files': {'dataset_name': 'credit_risk_dataset.csv',
  'interim': {'X_train': 'X_train.pkl',
   'y_train': 'y_train.pkl',
   'X_valid': 'X_valid.pkl',
   'y_valid': 'y_valid.pkl',
   'X_test': 'X_test.pkl',
   'y_test': 'y_test.pkl'}},
 'path_raw_dir': '/home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/raw',
 'path_interim_dir': '/home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim',
 'path_processed_dir': '/home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/processed',
 'path_raw_data': '/home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/raw/credit_risk_dataset.csv',
 'path_X_train': '/home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/X_train.pkl',
 'path_y_train': '/home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/y_train.pkl',
 'path_X_valid': '/home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO

In [4]:
def load_data(fname: str) -> pd.DataFrame:
    """
    Memuat dataset dari file csv ke dalam pandas dataframe.

    Args:
        fname (str): lokasi file (path) .csv 

    Returns: 
        pd.DataFrame: DataFrame yang berisi data dari file CSV tersebut
    """
    if not os.path.exists(fname):
        raise FileNotFoundError(f"File tidak ditemukan di {fname}")

    data = pd.read_csv(fname)
    print(f"Data Shape: {data.shape}")

    return data

In [5]:
FNAME = config["path_raw_data"]
data = load_data(FNAME)

Data Shape: (32581, 12)


In [6]:
data.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [7]:
def split_input_output(data: pd.DataFrame, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Memisahkan dataset menjadi fitur (X) dan target (y)

    Fungsi ini menghapus kolom target dari dataframe utama untuk membuat
    kumpulan fitur, dan mengekstrak kolom target tersebut secara terpisah 

    Args:
        data (pd.DataFrame): Dataframe asli (raw) yang berisi fitur dan target
        target_col (str): Nama kolom yang akan dijadikan label/target

    Returns:
        Tuple[pd.DataFrame, pd.Series]:
            - X (pd.DataFrame): fitur (semua kolom kecuali target)
            - y (pd.Series): target (hanya kolom target)
    """

    X = data.drop(target_col, axis=1)
    y = data[target_col]

    print(f"Original data shape: {data.shape}")
    print(f"X data shape       : {X.shape}")
    print(f"y data shape       : {y.shape}")

    return X, y

In [8]:
TARGET_COL = "loan_status"
X, y = split_input_output(data, TARGET_COL)

Original data shape: (32581, 12)
X data shape       : (32581, 11)
y data shape       : (32581,)


In [9]:
X.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,0.55,Y,4


In [10]:
y.head()

0    1
1    0
2    1
3    1
4    1
Name: loan_status, dtype: int64

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
def split_train_test(
        X: pd.DataFrame,
        y: pd.Series,
        test_size: float,
        random_state: int
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    Membagi data menjadi training set dan testing set
    Fungsi ini menggunakan train_test_split dari sklearn untuk memastikan 
    konsistensi random state

    Args:
        X (pd.DataFrame): Fitur
        y (pd.Series): Target
        test_size (float): Proporsi data untuk test set (0.0 - 1.0)
        random_state (int): Seed untuk reproduksibilitas hasil

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
            Urutannya adalah: (X_train, X_test, y_train, y_test)
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    print(f"X train shape: {X_train.shape}")
    print(f"X test shape : {X_test.shape}")
    print(f"y train shape: {y_train.shape}")
    print(f"y test shape : {y_test.shape}")

    return X_train, X_test, y_train, y_test

In [13]:
X_train, X_non_train, y_train, y_non_train = split_train_test(
    X, y,
    test_size=0.2,
    random_state=42
)

X_valid, X_test, y_valid, y_test = split_train_test(
    X=X_non_train,
    y=y_non_train,
    test_size=0.5,
    random_state=42
)

X train shape: (26064, 11)
X test shape : (6517, 11)
y train shape: (26064,)
y test shape : (6517,)
X train shape: (3258, 11)
X test shape : (3259, 11)
y train shape: (3258,)
y test shape : (3259,)


In [14]:
import joblib

In [15]:
def serialize_data(data: Any, path: str) -> None:
    """
    Menyimpan python object ke dalam file serealisasi 
    Fungsi ini menggunakan library joblib untuk mengubah objek seperti
    DataFrame atau Model menjadi file .pkl agar bisa digunakan kembali
    tanpa harus memproses ulang dari awal.

    Args:
        data (Any): Objek yang ingin disimpan, bisa berupa DataFrame, list, atau model
        path (str): Lokasi dan nama file tujuan
    """

    os.makedirs(os.path.dirname(path), exist_ok=True)
    
    joblib.dump(data, path)
    print(f"Sukses menyimpan data ke: {path}")


In [16]:
serialize_data(X_train, config["path_X_train"])
serialize_data(X_valid, config["path_X_valid"])
serialize_data(X_test, config["path_X_test"])
serialize_data(y_train, config["path_y_train"])
serialize_data(y_valid, config["path_y_valid"])
serialize_data(y_test, config["path_y_test"])

Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/X_train.pkl
Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/X_valid.pkl
Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/X_test.pkl
Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/y_train.pkl
Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/y_valid.pkl
Sukses menyimpan data ke: /home/amadeo/Pacmann/MLProcess/Mentoring/AMADEO_MLPROCESS/data/interim/y_test.pkl


In [17]:
def deserialize_data(path: str) -> Any:
    """
    Memuat kembali objek yang telah disimpan dari hasil serealisasi
    Fungsi ini membaca file .pkl yang ada di folder dan mengembalikannya
    ke dalam variabel python

    Args:
        path (str): Lokasi file .pkl yang ingin dimuat

    Returns:
        Any: Objek asli yang sebelumnya disimpan 
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"File .pkl tidak ditemuka di {path}")

    data = joblib.load(path)
    return data


In [18]:
X_coba = deserialize_data(path=config["path_X_train"])

In [19]:
X_coba.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
15884,25,241875,MORTGAGE,4.0,EDUCATION,A,16000,7.05,0.07,N,4
15138,21,18000,RENT,5.0,PERSONAL,B,1500,12.18,0.08,N,4
7474,25,53000,MORTGAGE,10.0,MEDICAL,B,16000,12.53,0.30,N,2
18212,28,16800,OWN,NaN,MEDICAL,C,5000,13.98,0.30,N,8
6493,25,50000,MORTGAGE,2.0,VENTURE,A,10000,7.90,0.20,N,2
